# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [1]:
%pip install -q --upgrade "mlflow[databricks]"

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for whenever (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [81 lines of output]
      C:\Users\alfes\AppData\Local\Temp\pip-build-env-xbcx0q91\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ************************************************

In [2]:
import json
from typing import Any

import mlflow

USE_LLM = False
AGENT_VERSION = "v1-deterministic"
EXPERIMENT_NAME = "/Shared/muiaap-operacion-modelos/semana01-agentops"
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})

2026/10/05 17:59:53 INFO mlflow.tracking.fluent: Experiment with name '/Shared/muiaap-operacion-modelos/semana01-agentops' does not exist. Creating a new experiment.


## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados. La ruta security activa terminos relacionados con tokens o contraseñas. Production relacionado con produccion, Apis, y el resto de preguntas se van a mlflow

In [3]:
@mlflow.trace
def route_question(question: str) -> str:
    text = question.lower()
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str:
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

[{'question': '¿Qué objeto agrupa varios runs?',
  'answer': 'Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.'},
 {'question': '¿Puedo pegar mi token en un parámetro de MLflow?',
  'answer': 'Ruta: security. No registres secretos en parámetros, tags ni artefactos.'},
 {'question': '¿Qué cambia cuando llevo un modelo a producción?',
  'answer': 'Ruta: production. En producción importan contratos, versionado y monitorización.'}]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.


## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.


In [4]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    # TODO 2: añade dos casos más y comprueba que siguen pasando.
    {"question": "¿Se puede guardar en Mlflow un token secreto?", "must_include": "secretos"},
    {"question": "¿Qué tengo que tener en cuenta al desplegar una API en produccion?", "must_include": "monitorización"},
]

evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

,question,must_include,answer,passed
0,¿Qué objeto agrupa varios runs?,experimento,Ruta: mlflow. Un experimento agrupa runs con p...,True
1,¿Puedo guardar una contraseña como tag?,secretos,Ruta: security. No registres secretos en parám...,True
2,¿Qué debo cuidar en producción?,contratos,Ruta: production. En producción importan contr...,True
3,¿Se puede guardar en Mlflow un token secreto?,secretos,Ruta: security. No registres secretos en parám...,True
4,¿Qué tengo que tener en cuenta al desplegar un...,monitorización,Ruta: production. En producción importan contr...,True


## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.


In [6]:
mlflow.end_run()

In [7]:
with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df))
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean()))
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

{'evaluation_run_id': '3462854638ca4c3bad35ead25b5e8fcf', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.


In [8]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

,run_id,tags.agent.version,metrics.pass_rate
0,3462854638ca4c3bad35ead25b5e8fcf,v1-deterministic,1.0
1,ae166abec09c42a890473e04772259a3,v1-deterministic,NaN


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente.
